# Comparação estatística LSTM vs BiLSTM (RUL FD001)

Carrega as tabelas gravadas por src/lstm.ipynb e src/bilstm.ipynb e aplica o teste de Wilcoxon pareado. Execute os dois notebooks de modelo antes deste.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "data.py").is_file():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import (
    compare_models_wilcoxon,
    load_experiment_results,
    summarize_experiment_results,
)

RESULTS_DIR = PROJECT_ROOT / "data" / "processed" / "results"
RESULTS_LSTM = RESULTS_DIR / "lstm_results.csv"
RESULTS_BILSTM = RESULTS_DIR / "bilstm_results.csv"


## Tabelas salvas e teste de Wilcoxon

In [ ]:
missing = [path for path in (RESULTS_LSTM, RESULTS_BILSTM) if not path.is_file()]
if missing:
    names = ", ".join(str(path) for path in missing)
    raise FileNotFoundError(
        "Execute src/lstm.ipynb e src/bilstm.ipynb antes desta comparação. "
        f"Arquivos ausentes: {names}"
    )

resultados_lstm = load_experiment_results(RESULTS_LSTM)
resultados_bilstm = load_experiment_results(RESULTS_BILSTM)

print("Resultados LSTM".center(40, "-"))
print(f"{resultados_lstm.to_string(index=False)}\n")
print("Resultados BiLSTM".center(40, "-"))
print(f"{resultados_bilstm.to_string(index=False)}\n")

resumo_lstm = summarize_experiment_results(resultados_lstm)
resumo_bilstm = summarize_experiment_results(resultados_bilstm)

print("Resumo LSTM".center(40, "-"))
print(f"{resumo_lstm}\n")
print("Resumo BiLSTM".center(40, "-"))
print(f"{resumo_bilstm}\n")

wilcoxon_df = compare_models_wilcoxon(resultados_lstm, resultados_bilstm)
print("Teste de Wilcoxon".center(40, "-"))
for _, row in wilcoxon_df.iterrows():
    metrica = row["metrica"]
    valorp = row["p_value"]
    if row["significativo"]:
        conclusao = "A diferença é estatisticamente significativa"
    else:
        conclusao = "A diferença não é estatisticamente significativa"
    print(f"O p-value para {metrica} é {valorp}. {conclusao}")
